# FitCoach Streaming Live Feedback (Variable Timing)

This notebook implements live feedback with **variable timing** - the model decides when to speak, just like the original project.

**Key differences from lightweight version:**
- Model controls feedback timing (not fixed 15-second intervals)
- Uses `<vision>` and `<answer>` tokens for asynchronous generation
- More memory intensive but more natural coaching behavior

**Requirements:** GPU with at least 24GB VRAM (A100 recommended, T4 may struggle)

---

## Step 1: Check GPU

In [ ]:
!nvidia-smi

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {vram_gb:.1f} GB")
    
    if vram_gb >= 24:
        print(f"\nSufficient VRAM for streaming mode!")
    else:
        print(f"\nWARNING: Streaming mode may run out of memory on this GPU")
        print(f"   Recommended: Use FitCoach_Live_Feedback_Colab_Fixed.ipynb (lightweight mode) instead")
else:
    print("\nNo GPU! Go to Runtime -> Change runtime type -> GPU")

## Step 2: Clone Repository

In [ ]:
print("Cloning repository...\n")
!git clone -b kendrick/live-feedback https://github.com/KendrickXie/FitCoach.git
%cd FitCoach
print("\nRepository cloned!")

## Step 3: Install Dependencies

In [ ]:
print("Installing dependencies (3-5 minutes)...\n")
print("You may see warnings/errors - these are normal!\n")

# Core dependencies (required)
print("[1/12] Installing PyYAML...")
!pip install -q PyYAML==6.0

print("[2/12] Installing datasets...")
!pip install -q datasets==2.14.6

print("[3/12] Installing evaluate...")
!pip install -q evaluate==0.4.1

print("[4/12] Installing OpenCV...")
!pip install -q opencv-python==4.9.0.80

print("[5/12] Installing transformers...")
!pip install -q transformers==4.36.0

print("[6/12] Installing accelerate...")
!pip install -q accelerate==0.24.1

print("[7/12] Installing peft...")
!pip install -q peft==0.5.0

print("[8/12] Installing bitsandbytes (with CUDA support)...")
!pip install -q bitsandbytes>=0.44.0

print("[9/12] Installing tqdm...")
!pip install -q tqdm

print("[10/12] Installing rouge_score...")
!pip install -q rouge_score

print("[11/12] Installing bert_score...")
!pip install -q bert_score

# Fix NumPy compatibility (OpenCV requires NumPy 1.x)
print("[12/12] Fixing NumPy compatibility...")
!pip install -q "numpy<2"

# Flash attention (optional - skip if fails)
print("\n[Optional] Attempting flash-attn installation...")
print("(This may fail - it's OK, we'll use standard attention)\n")

import subprocess
try:
    result = subprocess.run(
        ['pip', 'install', '-q', 'flash-attn==2.5.8', '--no-build-isolation'],
        capture_output=True,
        timeout=300  # 5 minute timeout
    )
    if result.returncode == 0:
        print("Flash attention installed!")
    else:
        print("Flash attention failed (optional) - using standard attention")
except (subprocess.TimeoutExpired, Exception) as e:
    print("Flash attention skipped (optional) - using standard attention")

print("\n" + "="*60)
print("Setup complete! All required dependencies installed.")
print("="*60)
print("\nIMPORTANT: Go to Runtime -> Restart runtime")
print("After restart, skip Steps 1-3 and go directly to Step 4 (Download Models)")

## Step 4: Download Models

### 4a: Login to HuggingFace

**Required:** Get access to LLaMA-2
1. Go to: https://huggingface.co/meta-llama/Llama-2-7b-hf
2. Click "Request access" (instant approval)
3. Get token: https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import notebook_login

print("Please login with your HuggingFace token:")
print("Get token: https://huggingface.co/settings/tokens\n")

notebook_login()

### 4b: Download LLaMA-2-7B (~13GB, 5-8 minutes)

In [ ]:
from huggingface_hub import snapshot_download
from tqdm import tqdm

print("Downloading LLaMA-2-7B (~13GB)...")
print("This will take 5-8 minutes...\n")

snapshot_download(
    repo_id="meta-llama/Llama-2-7b-hf",
    local_dir="./Llama-2-7b-hf",
    local_dir_use_symlinks=False
)

print("\nLLaMA-2-7B downloaded!")

### 4c: Download 3D CNN weights (~500MB, 1 minute)

In [ ]:
print("📥 Downloading 3D CNN weights...\n")

!mkdir -p ckpts_efficientnet
!wget --no-check-certificate -P ./ckpts_efficientnet \
    https://github.com/Qualcomm-AI-research/FitCoach/releases/download/v1.0/efficientnet_3d_cnn_weights.tar.gz

print("\nExtracting (note: file is .tar not .tar.gz despite name)...")
!cd ckpts_efficientnet && tar -xf efficientnet_3d_cnn_weights.tar.gz

print("3D CNN weights ready!")

### 4d: Download Stream-VLM weights (~3.5GB, 3-5 minutes)

In [ ]:
print("Downloading Stream-VLM weights (6 parts)...\n")

!mkdir -p ckpts_streamvlm && cd ckpts_streamvlm && \
    echo "Downloading part 1/6..." && \
    wget --no-check-certificate https://github.com/Qualcomm-AI-research/FitCoach/releases/download/v1.0/streamvlm_weights.tar.gz.aa && \
    echo "Downloading part 2/6..." && \
    wget --no-check-certificate https://github.com/Qualcomm-AI-research/FitCoach/releases/download/v1.0/streamvlm_weights.tar.gz.ab && \
    echo "Downloading part 3/6..." && \
    wget --no-check-certificate https://github.com/Qualcomm-AI-research/FitCoach/releases/download/v1.0/streamvlm_weights.tar.gz.ac && \
    echo "Downloading part 4/6..." && \
    wget --no-check-certificate https://github.com/Qualcomm-AI-research/FitCoach/releases/download/v1.0/streamvlm_weights.tar.gz.ad && \
    echo "Downloading part 5/6..." && \
    wget --no-check-certificate https://github.com/Qualcomm-AI-research/FitCoach/releases/download/v1.0/streamvlm_weights.tar.gz.ae && \
    echo "Downloading part 6/6..." && \
    wget --no-check-certificate https://github.com/Qualcomm-AI-research/FitCoach/releases/download/v1.0/streamvlm_weights.tar.gz.af && \
    echo "Extracting..." && \
    cat streamvlm_weights.tar.gz.* | tar xzf - && \
    cd ..

print("\nStream-VLM weights ready!")
print("\n" + "="*60)
print("All models downloaded! Ready to process videos.")
print("="*60)

## Step 5: Upload Video

In [ ]:
from google.colab import files

print("Please upload your workout video:")
print("Supported: MP4, AVI, MOV")
print("Recommended: 30-120 seconds\n")
print("Note: Longer videos use more memory in streaming mode\n")

uploaded = files.upload()

if uploaded:
    video_filename = list(uploaded.keys())[0]
    print(f"\nVideo uploaded: {video_filename}")
else:
    print("\nNo video uploaded")
    video_filename = None

## Step 6: Run FitCoach Streaming!

**Set your exercise below:**

**How this works:**
- Model continuously processes video frames
- Generates `<vision>` tokens to "watch" without speaking
- Generates `<answer>` tokens when it decides to give feedback
- Variable timing - feedback appears when model thinks it's appropriate

In [ ]:
# ===== CONFIGURE HERE =====
exercise_type = "squats"  # Change to: squats, push-ups, jumping-jacks, etc.
# ==========================

if video_filename is None:
    print("No video! Upload one in Step 5 first.")
else:
    print("🚀 Starting FitCoach Streaming Feedback...\n")
    print(f"Video: {video_filename}")
    print(f"Exercise: {exercise_type}")
    print(f"Mode: streaming (variable timing)")
    print("\n" + "="*60 + "\n")
    
    config_file = "configs/live_streaming.yaml"
    script_file = "scripts/live_feedback_streaming.py"
    
    # Change to FitCoach directory
    import os
    os.chdir('/content/FitCoach')
    
    # Run the script with PYTHONPATH set
    !PYTHONPATH=/content/FitCoach python $script_file --config $config_file --video $video_filename --exercise $exercise_type --headless
    
    print("\n" + "="*60)
    print("✅ Processing complete!")
    print("="*60)

## Understanding the Output

You should see:
- `[Frame XXX] Coach is speaking...` - Model decided to give feedback
- `💬 Coach: <feedback message>` - The actual feedback
- Feedback appears at **variable times** based on what the model sees

**Comparison with lightweight mode:**
- **Lightweight**: Fixed 15-second intervals, predictable timing
- **Streaming**: Variable timing, more natural but uses more memory

---

## Troubleshooting

### "CUDA out of memory"
**Solution:** Streaming mode needs more VRAM than lightweight mode
1. Try shorter video (30-60 seconds)
2. Switch to lightweight mode notebook instead
3. Request A100 GPU if available

### "No feedback generated"
- Model may be watching without speaking (generating `<vision>` tokens)
- Try a video with clearer exercise form
- Check that exercise type matches video content

### "Too slow"
- Streaming mode is slower than lightweight due to continuous generation
- Expected: 2-3x video length processing time
- Use lightweight mode for faster processing